# Import Variables

In [1]:
import os
exec(open("./PGS_calc_param.txt").read())

# Set Up

In [2]:
import time
!pip install pyliftover
from pyliftover import LiftOver
import subprocess
from pathlib import Path
import pandas as pd
import gzip
from glob import glob
import psutil
import re

first_start = time.time()
start = time.time()
weights_dir = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_catalog_weights")
output_dir = Path("/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights")
weights_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
###########################################################################################
# 0. Grab BIM IDs
###########################################################################################

# Load all BIM files
bim_files = glob("/home/jupyter/workspace/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/plink_bed/chr*.bim")

# Combine bim files into single file
print("Loading BIM files...")
bim = pd.concat(
    [pd.read_csv(f, sep="\t", header=None) for f in bim_files],
    ignore_index=True
)

# Set of BIM SNP IDs for fast lookup
bim_ids = set(bim.iloc[:,1])
del bim # reduce memory usage

##### track time #####
print(f"Loaded {len(bim_ids):,} variants ({time.time()-start:.1f}s)")
######################

Loading BIM files...
Loaded 111,404,689 variants (129.9s)


# Download raw PGS files from PGS catalog

In [3]:
raw_files = {}
for pgs_id in PGS_ID:
    print(pgs_id)
    url = f"https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/{pgs_id}/ScoringFiles/Harmonized/{pgs_id}_hmPOS_{BUILD}.txt.gz"
    raw_file = weights_dir / f"{pgs_id}_hmPOS_{BUILD}.txt.gz"
    if not raw_file.exists() or raw_file.stat().st_size == 0:
        raw_file.unlink(missing_ok=True)
        print(f"Downloading {pgs_id}...")
        max_download_attempts = 6
        for attempt in range(1, max_download_attempts + 1):
            try:
                subprocess.run(
                    ["curl", "-fL", "--connect-timeout", "60", "--max-time", "3600",
                     url, "-o", str(raw_file)],
                    check=True,
                )
                break
            except subprocess.CalledProcessError as exc:
                raw_file.unlink(missing_ok=True)
                if attempt == max_download_attempts:
                    raise RuntimeError(
                        f"Download failed for {pgs_id} after {max_download_attempts} attempts"
                    ) from exc
                retry_wait = min(5 * (2 ** (attempt - 1)), 120)
                print(
                    f"{pgs_id}: download attempt {attempt}/{max_download_attempts} failed; "
                    f"retrying in {retry_wait}s ({exc})"
                )
                time.sleep(retry_wait)
            except Exception:
                raw_file.unlink(missing_ok=True)
                raise
        if not raw_file.exists() or raw_file.stat().st_size == 0:
            raw_file.unlink(missing_ok=True)
            raise RuntimeError(f"Download produced no file for {pgs_id}")
    raw_files[pgs_id] = raw_file


PGS000014


# Script

## Function

In [4]:
def process_pgs(pgs_id):
    inital_start = time.time()
    start = time.time()
    
    ##################################################
    # 1. Load in weights file from PGS Catalog
    ##################################################
    print("########################## Starting", pgs_id, "#############################")
    
    raw_file = weights_dir / f"{pgs_id}_hmPOS_{BUILD}.txt.gz"
    # Read into enviornment 
    score_df = pd.read_csv(raw_file, sep="\t", comment="#", compression="gzip", low_memory=False)
    raw_score_length = len(score_df)

    # Use hm_inferOtherAllele where other_allele is absent or missing.
    if "other_allele" not in score_df.columns:
        if "hm_inferOtherAllele" not in score_df.columns:
            raise KeyError(
                f"{pgs_id}: neither 'other_allele' nor 'hm_inferOtherAllele' is available"
            )
        score_df["other_allele"] = score_df["hm_inferOtherAllele"]
        print(f"{pgs_id}: Other_allele column missing, used hm_inferOtherAllele")
    else:
        missing_other = (
            score_df["other_allele"].isna()
            | score_df["other_allele"].astype("string").str.strip().eq("")
        )
    
        if missing_other.any():
            if "hm_inferOtherAllele" not in score_df.columns:
                raise KeyError(
                    f"{pgs_id}: 'other_allele' has missing values, "
                    "but 'hm_inferOtherAllele' is unavailable"
                )
    
            score_df.loc[missing_other, "other_allele"] = score_df.loc[
                missing_other, "hm_inferOtherAllele"
            ]
    
        print(
            f"{pgs_id}: filled {missing_other.sum()} missing other_allele values "
            "from hm_inferOtherAllele"
        )
    
    ##################################################
    # 2. Liftover score file to GRCh38 as needed
    ##################################################

    ##### Extract genome build of score file from meta-data #####

    genome_build = None
    lo = LiftOver("hg19", "hg38")
    
    with gzip.open(raw_file, "rt") as f:
        for line in f:
            if line.startswith("#HmPOS_build="):
                genome_build = line.strip().split("=")[1]
                break
    
    print(pgs_id, ": Genome build:", genome_build)
    
    original_genome_build = None
    
    with gzip.open(raw_file, "rt") as f:
        for line in f:
            if line.startswith("#genome_build="):
                original_genome_build = line.strip().split("=")[1]
                break
    
    print(pgs_id, "Original Genome Build:", original_genome_build)
    
    source_build = original_genome_build.strip().lower()
    check_strand_flips = source_build in {"grch37", "hg19", "grch37/hg19", "hg19/grch37"}
    
    
    ##### Perform liftover if needed #####
    build = genome_build.strip().lower()
    needs_liftover = build in ["grch37", "hg19", "grch37/hg19", "hg19/grch37"]

    
    if needs_liftover:
    
        print("Lifting over from ", genome_build, "to GRCh38")
        lo = LiftOver("hg19", "hg38")
    
        def liftover_pos(chrom, pos):
            result = lo.convert_coordinate(f"chr{chrom}", int(pos))
            if not result:
                return None, None, None
    
            new_chr, new_pos, strand, _ = result[0]
            return new_chr.removeprefix("chr"), new_pos, strand
    
        score_df[["chr_name_38", "chr_position_38"]] = score_df.apply(
            lambda row: liftover_pos(row["hm_chr"], row["hm_pos"]),
            axis=1,
            result_type="expand"
        )
    
        score_df["chr_position_38"] = pd.to_numeric(
            score_df["chr_position_38"],
            errors="coerce"
        ).astype("Int64")
    
    else:
        print("No liftover needed, already in GRCh38")
        # No liftover needed — just copy original coordinates
        score_df["chr_name_38"] = score_df["hm_chr"]
        score_df["chr_position_38"] = pd.to_numeric(
            score_df["hm_pos"],
            errors="coerce"
        ).astype("Int64")
    
    
    with gzip.open(raw_file, "rt") as f:
        for line in f:
            if line.startswith("#genome_build="):
                original_genome_build = line.strip().split("=")[1]
                break
    
    print(pgs_id, "Original Genome Build:", original_genome_build)
    
    remove_ambiguous = pd.Series(False, index=score_df.index)
    strand_flip = pd.Series(False, index=score_df.index)
    
    if check_strand_flips:
        print("Original build is GRCh37; checking liftover strand")
    
        # Use the original GRCh37 coordinates for strand lookup
        source_chr = "chr_name"
        source_pos = "chr_position"
    
        def get_strand_info(chrom, pos):
            if pd.isna(chrom) or pd.isna(pos):
                return None

            result = lo.convert_coordinate(f"chr{chrom}", int(pos))
            if not result:
                return None

            _, _, strand, _ = result[0]
            return strand
    
        score_df["liftover_strand"] = score_df.apply(
            lambda row: get_strand_info(row[source_chr], row[source_pos]),
            axis=1,
        )
    
        strand = score_df["liftover_strand"]
        
        effect = score_df["effect_allele"].astype("string").str.upper()
        other = score_df["other_allele"].astype("string").str.upper()
        palindromic = (
            (effect.eq("A") & other.eq("T"))
            | (effect.eq("T") & other.eq("A"))
            | (effect.eq("C") & other.eq("G"))
            | (effect.eq("G") & other.eq("C"))
        )

        strand = score_df["liftover_strand"]
        remove_ambiguous = strand.isna() & palindromic

        print(
            f"Removing {remove_ambiguous.sum()} palindromic rows "
            "with NA liftover strand due to ambiguity"
        )
        score_df = score_df.loc[~remove_ambiguous].copy()

        strand_flip = score_df["liftover_strand"].eq("-") & palindromic
        print(f"{pgs_id}: {strand_flip.sum()} palindromic rows flipped")
    
        old_effect = score_df.loc[strand_flip, "effect_allele"].copy()
        score_df.loc[strand_flip, "effect_allele"] = score_df.loc[strand_flip, "other_allele"].values
        score_df.loc[strand_flip, "other_allele"] = old_effect.values
        score_df.loc[strand_flip, "effect_weight"] *= -1
        
    print(f"{pgs_id}: Weight file on GRCh38 loaded in ({time.time()-start:.1f}s)")

    ##################################################
    # 3.  Prepare SNP IDs to match .bim file
    ##################################################
    start = time.time()
    print("Starting harmonization...")

    ##########################################################################################
    ### Build forward and reverse IDs from PGS catalog score file
    ##########################################################################################

    # Build forward and reverse IDs
    print("Building SNP IDs...")
    score_df['SNP_ID'] = (
        "chr" + score_df['chr_name'].astype(str)
        + ":" + score_df['chr_position_38'].astype(str)
        + ":" + score_df['effect_allele']
        + ":" + score_df['other_allele']
    )

    score_df['SNP_rev'] = (
        "chr" + score_df['chr_name'].astype(str)
        + ":" + score_df['chr_position_38'].astype(str)
        + ":" + score_df['other_allele']
        + ":" + score_df['effect_allele']
    )

    ##### track time #####
    print(f"{pgs_id}: Done building SNP IDs in ({time.time()-start:.1f}s)")
    ######################

    #########################################################################################
    ### Find allele matches from PGS score file in BIM file
    ##########################################################################################

    start = time.time()
    print(pgs_id, ": Determining matches...")

    # Check both allele orientations in one membership pass, then split the
    # Boolean results back into forward and reverse matches.
    n_variants = len(score_df)
    candidate_ids = pd.concat(
        [score_df["SNP_ID"], score_df["SNP_rev"]],
        ignore_index=True,
    )
    candidate_matches = candidate_ids.isin(bim_ids).to_numpy()
    forward = pd.Series(candidate_matches[:n_variants], index=score_df.index)
    reverse = pd.Series(candidate_matches[n_variants:], index=score_df.index)
    del candidate_ids, candidate_matches
    flip = (~forward) & reverse
    matched = forward | reverse
    final = score_df.loc[matched].copy()

    ##########################################################################################
    #### Create harmonized score file
    ##########################################################################################
    # Flip effect weights where necessary
    print(pgs_id, ": Applying allele flips...")
    
    final.loc[flip.loc[matched], "effect_weight"] *= -1
    final.loc[flip.loc[matched], "effect_allele"] = score_df.loc[flip, "other_allele"].values
    final.loc[flip.loc[matched], "other_allele"] = score_df.loc[flip, "effect_allele"].values
    final.loc[flip.loc[matched], "SNP_ID"] = score_df.loc[flip, "SNP_rev"].values

    #### Final duplicate safety drop ####
    final = final.drop_duplicates(
        subset=["SNP_ID", "effect_allele", "effect_weight"],
        keep="first",)

    #### Weird case where SNP is duplicated with two different effect weights ####
    final = (
        final.groupby(["SNP_ID", "effect_allele", "other_allele"], as_index=False)
        .agg(effect_weight=("effect_weight", "sum")))

    ##### track time #####
    print(f"{pgs_id}: Flips applied ({time.time()-start:.1f}s)")
    print(f"{pgs_id}: Total orientation matches: {int(matched.sum())}")

    #########################################################################################
    ##### Save new PRS file
    ##########################################################################################
    
    # Select columns for PLINK score file
    plink_score_df = final[['SNP_ID', 'effect_allele', 'effect_weight']]

    # Save
    plink_score_file = output_dir / f"{pgs_id}_plink_score_{BUILD}_prepared.txt"
    plink_score_df.to_csv(
        plink_score_file,
        sep="\t",
        index=False,
        header=False
    )

    print(f"{pgs_id}: Saved {len(plink_score_df):,} variants to: {plink_score_file}")

    #########################################################################################
    ##### Save QC metrics file
    ##########################################################################################
    qc_file = output_dir / f"{pgs_id}_qc_metrics_{BUILD}.txt"
    n_total = len(score_df)
    qc_metrics = {
        "PGS_ID": pgs_id,
        "BUILD": BUILD,
        "genome_build_from_metadata": genome_build,
        "needs_liftover": needs_liftover,
        "total_pgs_variants_in_raw_score_file": raw_score_length,
        "sites_removed_due_to_strand_ambiguity": int(remove_ambiguous.sum()),
        "palindromic_sites_strand_flipped": int(strand_flip.sum()),
        "forward_matches": int(forward.sum()), 
        "reverse_matches": int(reverse.sum()),
        "both_orientation_matches": int((forward & reverse).sum()), 
        "flipped_variants": int(flip.sum()),
        "matched_variants": int(matched.sum()), 
        "unmatched_variants": int((~matched).sum()),
        "percent_found": round(100 * matched.sum() / n_total, 2) if n_total else 0,
        "variants_saved_to_plink_score_file": len(plink_score_df),
        "plink_score_file": plink_score_file,
        "qc_file": qc_file,
        "total_runtime_minutes": round((time.time() - inital_start) / 60, 2),
    }

    pd.DataFrame([qc_metrics]).to_csv(
        qc_file,
        sep="\t",
        index=False
    )

    print(pgs_id, ": Saved QC metrics to:")
    print(qc_file)

    ##### track time #####
    print(f"{pgs_id}: Done; Total time: {(time.time() - inital_start)/60:.1f} minutes")
    ######################

    return pgs_id, len(final), str(plink_score_file), str(qc_file)


# Run

In [5]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

pgs_ids = list(dict.fromkeys(PGS_ID))
results = []
errors = []

with ProcessPoolExecutor(
    max_workers=max(len(pgs_ids), 32),
    mp_context=mp.get_context("fork"),
) as executor:
    futures = {executor.submit(process_pgs, pgs_id): pgs_id for pgs_id in pgs_ids}

    for future in as_completed(futures):
        pgs_id = futures[future]
        try:
            result = future.result()
            results.append(result)
            print(f"{pgs_id}: saved {result[1]:,} matched variants to {result[2]}")
        except Exception as exc:
            errors.append((pgs_id, exc))
            print(f"{pgs_id}: FAILED: {exc}")


# Preserve successful IDs for downstream notebook steps and keep failures separate.
PGS_failed = [pgs_id for pgs_id, _ in errors]
PGS_IDS = [pgs_id for pgs_id in PGS_ID if pgs_id not in PGS_failed]

# Update param file
param_file = Path("./PGS_calc_param.txt")

param_text = param_file.read_text()
outcome_assignments = {
    "PGS_ID_updated": PGS_IDS,
    "PGS_failed": PGS_failed,
}
for name, value in outcome_assignments.items():
    assignment = f"{name} = {value!r}"
    pattern = rf"(?m)^{re.escape(name)}\s*=.*$"
    if re.search(pattern, param_text):
        param_text = re.sub(pattern, lambda _: assignment, param_text)
    else:
        param_text = param_text.rstrip() + "\n" + assignment + "\n"
param_file.write_text(param_text)
print(f"Updated parameter file: {param_file.resolve()}")

# Combine the individual per-PGS QC reports and display the summary.
pgs_ids_tag = "_".join(PGS_IDS)
combined_qc_file = output_dir / f"combined_qc_metrics_{pgs_ids_tag}.txt"
qc_files = [output_dir / f"{pgs_id}_qc_metrics_{BUILD}.txt" for pgs_id in PGS_IDS]
combined_qc = pd.concat(
    [pd.read_csv(qc_file, sep="\t") for qc_file in qc_files],
    ignore_index=True,
)
combined_qc.to_csv(combined_qc_file, sep="\t", index=False)
print(f"Finished {len(results)} PGS jobs in  {(time.time() - first_start)/60:.1f} minutes")
print(f"Combined QC metrics saved to: {combined_qc_file}")

# Combine prepared PLINK score rows, retaining their PGS_ID for downstream use.
combined_weights_file = output_dir / f"combined_plink_score_{pgs_ids_tag}_{BUILD}_prepared.txt"
weight_frames = []
for pgs_id in PGS_IDS:
    weights_file = output_dir / f"{pgs_id}_plink_score_{BUILD}_prepared.txt"
    weights = pd.read_csv(
        weights_file,
        sep="\t",
        header=None,
        names=["SNP_ID", "effect_allele", "effect_weight"],
    )
    weights.insert(0, "PGS_ID", pgs_id)
    weight_frames.append(weights)
combined_weights = pd.concat(weight_frames, ignore_index=True)
combined_weights.to_csv(combined_weights_file, sep="\t", index=False)
print(f"Combined weights saved to: {combined_weights_file}")

# Add combined weights file to param file
param_file = Path("./PGS_calc_param.txt")
param_text = param_file.read_text()

name = "COMBINED_WEIGHTS_FILE"
assignment = f"{name} = {str(combined_weights_file)!r}"
pattern = rf"(?m)^{name}\s*=.*$"

if re.search(pattern, param_text):
    param_text = re.sub(pattern, lambda _: assignment, param_text)
else:
    param_text = param_text.rstrip() + "\n" + assignment + "\n"

param_file.write_text(param_text)
print(f"Updated parameter file: {param_file.resolve()}")

combined_qc.head(20)

########################## Starting PGS000014 #############################
PGS000014: filled 0 missing other_allele values from hm_inferOtherAllele
PGS000014 : Genome build: GRCh38
PGS000014 Original Genome Build: hg19
No liftover needed, already in GRCh38
PGS000014 Original Genome Build: hg19
Original build is GRCh37; checking liftover strand
Removing 0 palindromic rows with NA liftover strand due to ambiguity
PGS000014: 0 palindromic rows flipped
PGS000014: Weight file on GRCh38 loaded in (114.9s)
Starting harmonization...
Building SNP IDs...
PGS000014: Done building SNP IDs in (12.4s)
PGS000014 : Determining matches...
PGS000014 : Applying allele flips...
PGS000014: Flips applied (184.2s)
PGS000014: Total orientation matches: 6853550
PGS000014: Saved 6,853,545 variants to: /home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/PGS000014_plink_score_GRCh38_prepared.txt
PGS000014 : Saved QC metrics to:
/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs

,PGS_ID,BUILD,genome_build_from_metadata,needs_liftover,total_pgs_variants_in_raw_score_file,sites_removed_due_to_strand_ambiguity,palindromic_sites_strand_flipped,forward_matches,reverse_matches,both_orientation_matches,flipped_variants,matched_variants,unmatched_variants,percent_found,variants_saved_to_plink_score_file,plink_score_file,qc_file,total_runtime_minutes
0,PGS000014,GRCh38,GRCh38,False,6917436,0,0,3367422,3486128,0,3486128,6853550,63886,99.08,6853545,/home/jupyter/workspace/workspace-bucket/calcu...,/home/jupyter/workspace/workspace-bucket/calcu...,5.58


In [6]:
import pandas as pd
check = pd.read_csv("/home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/PGS000014_plink_score_GRCh38_prepared.txt",
                   sep="\t")
check.head(5)


                  

,chr10:100000235:C:T,C,2.1313e-06
0,chr10:100000943:G:A,G,0.000005
1,chr10:100000979:T:C,T,-0.000007
2,chr10:100002012:T:C,T,0.000022
3,chr10:100002628:A:C,A,0.000001
4,chr10:100002875:A:G,A,-0.000006


In [7]:
check[check.iloc[:, 0] == "chr7:495710:A:G"]

,chr10:100000235:C:T,C,2.1313e-06
6053753,chr7:495710:A:G,A,0.000009
